## Current project status — final test evaluation complete, 3 October 2026

The original, morning-best trial 13 and boundary-best epoch 4 models are trained and preserved. **Morning best is locked and has been evaluated on all 689 test pairs: road IoU 91.65%, Dice 95.65%.** See the [final test report](runs/test_evaluation_20261003_morning_best/TEST_REPORT.md), [test gallery](runs/test_evaluation_20261003_morning_best/gallery/index.html), [model catalog](models/README.md) and [project index](README.md).

The setup explanations and code cells below describe the recorded construction/training workflow. This source notebook has no saved execution output. They were not rerun during final evaluation. The disposable untrained setup binary was removed; setup cells can rebuild a fresh decoder from retained ImageNet weights. `RUN_TRAINING` remains false. Read the final result section at the end for the current status.

The preprocessing notebook now previews train/validation by default; its preprocessing semantics are preserved; execution output is omitted from the source checkout. Split **5507/687/689**, normalization once to **[-1,1]** and masks **0/1** remain unchanged. The test set has now been used for reporting; future test-informed model changes need independent evaluation data.


# U-Net with a pretrained MobileNetV2 encoder

Our project: **Adaptive Path Planning and Collision Avoidance for Autonomous Vehicles on Unstructured Indian Roads**.

This notebook describes the learned road-perception module. It reuses the completed `idd_data.py` pipeline and preserves **5,507 train / 687 val / 689 test** pairs. Images are already normalized to **[-1, 1]** and masks are **0/1**. There is no second normalization.

The following construction cells are historical teaching records. The former `UNTRAINED` checkpoint had an ImageNet encoder and fresh decoder; it was removed during cleanup. Use the locked morning-best model for current inference.

Run from the project root. This notebook uses `idd_model.py`, `idd_data.py`, `train_idd.py`, and the `models` setup artifacts. For a later Colab/WSL run, copy the project code, initial model/configuration, and the existing complete split folder without changing its manifest.


## 1. Settings and compute
An encoder extracts image features. A decoder turns them into a pixel map. Transfer learning reuses ImageNet-trained encoder weights.

The recorded GPU training used a verified WSL2 environment. Select and verify the intended CPU/GPU runtime explicitly before enabling training. **RUN_TRAINING defaults to False.**


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import tensorflow as tf

PROJECT_DIR = Path.cwd()
SPLITS_DIR = PROJECT_DIR / "dataset" / "splits"
INITIAL_MODEL = None  # Fresh setup model can be rebuilt from cached ImageNet weights.
RUN_TRAINING = False
ALLOW_CPU_TRAINING = False
EPOCHS = 20
BATCH_SIZE = 4
SEED = 42

sys.path.insert(0, str(PROJECT_DIR))
from idd_data import read_pairs, make_dataset
from idd_model import get_encoder_weights, build_unet, compile_model, BCEDiceLoss

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Full training enabled:", RUN_TRAINING)


## 2. Preserve the existing split
The manifest associates image and mask by ID. This reads metadata/file names and checks the counts. Test images will not be fed through the model here.


In [ ]:
pairs = read_pairs(SPLITS_DIR)
counts = {split: len(records) for split, records in pairs.items()}
assert counts == {"train": 5507, "val": 687, "test": 689}
print("Cleaned split:", counts)


## 3. Load the pretrained encoder and verified initial model
The official ImageNet weights are cached inside `models/cache/weights`. Loading a cache does not require another download. The U-Net's verified initial checkpoint already contains those weights.

`include_top=False` removes MobileNetV2's original image-classification head. The no-top convolutional weights can process 512 x 512 input; the `224` in the archive name does not mean we resize the dataset.


In [ ]:
weights_path = get_encoder_weights(PROJECT_DIR / "models" / "cache")
if INITIAL_MODEL is not None and INITIAL_MODEL.is_file():
    model = tf.keras.models.load_model(INITIAL_MODEL, compile=False)
else:
    model = build_unet(weights_path)
compile_model(model, learning_rate=1e-3)
print("Encoder archive:", weights_path.name)
print("Model input:", model.input_shape)
print("Model output:", model.output_shape)
print("Encoder frozen:", not model.get_layer("mobilenetv2_encoder").trainable)
print("Parameters:", f"{model.count_params():,}")


## 4. Understand the U shape
The encoder reduces spatial resolution: **512 -> 256 -> 128 -> 64 -> 32 -> 16**.
The decoder expands it: **16 -> 32 -> 64 -> 128 -> 256 -> 512**.
At four decoder levels, skip connections join the corresponding encoder features, helping recover boundary details. They never use the ground-truth mask as model input.

The last sigmoid layer gives one 0-to-1 road probability per pixel. A probability is the model's estimate, not a steering command.


In [ ]:
encoder = model.get_layer("mobilenetv2_encoder")
for tensor in encoder.outputs:
    print("Encoder feature shape:", tuple(tensor.shape))
print("Final road-probability shape:", model.output_shape)


## 5. Check one real forward pass
A forward pass sends an image through the network. The existing loader supplies normalized images and binary masks. The model produces probabilities. A matching shape and finite output verify its wiring; they do not establish good road segmentation.

BCE compares each probability with its label. Soft Dice compares road-region overlap. We add them with equal weight. No prediction threshold is applied in the differentiable loss.


In [ ]:
setup_ds = make_dataset(pairs["train"], training=True, batch_size=BATCH_SIZE, seed=SEED)
images, masks = next(iter(setup_ds))
probabilities = model(images, training=False)
loss_value = BCEDiceLoss()(masks, probabilities)
assert images.shape == (4, 512, 512, 3)
assert probabilities.shape == masks.shape == (4, 512, 512, 1)
assert bool(tf.reduce_all(tf.math.is_finite(probabilities)))
assert bool(tf.math.is_finite(loss_value))
print("Images:", images.shape, "Masks/output:", masks.shape)
print("Probability range:", float(tf.reduce_min(probabilities)), float(tf.reduce_max(probabilities)))
print("Setup loss (untrained decoder):", float(loss_value))
print("Optimizer updates:", int(model.optimizer.iterations.numpy()))


## 6. Read the completed setup checks
The local verification checked finite nonzero decoder gradients without optimizer updates, unchanged frozen encoder weights, hand-calculated loss/IoU/Dice fixtures, and save/reload equivalence. The initial checkpoint was saved before the temporary gradient probe.

Road IoU and Dice accumulate TP/FP/FN over all pixels and batches at threshold 0.5. An entirely empty evaluated union receives score 1. These are global class-1 road metrics, not per-image averages or binary mean IoU.


In [ ]:
verification = json.loads((PROJECT_DIR / "models" / "model_verification.json").read_text())
print("Verification status:", verification["status"])
for name, result in verification["checks"].items():
    print(name, result)
print("Segmentation training started:", verification["segmentation_training_started"])


## 7. Historical training setup — execution disabled by default
The proposed initial run freezes the encoder and learns the decoder with Adam at 0.001, at most 20 epochs, batch size 4, and seed 42. The best checkpoint maximizes validation road IoU. Early stopping and learning-rate reduction monitor validation progress. Logs and versions are saved.

An epoch is a complete pass through training data. Validation monitors progress without learning from its labels. At the time of this historical run, test data was reserved. The locked final test evaluation is now complete; do not use its results to choose another model without new independent evaluation data. Later fine-tuning of selected encoder blocks at a lower learning rate is conditional on validation review; it is not automatic.

After choosing and verifying a GPU runtime, set `RUN_TRAINING=True` to use this cell. The run directory must be new. CPU training additionally requires `ALLOW_CPU_TRAINING=True` as an explicit choice.


In [ ]:
if RUN_TRAINING:
    from train_idd import train
    history = train(
        SPLITS_DIR, INITIAL_MODEL, PROJECT_DIR / "runs" / "frozen_new_01",
        epochs=EPOCHS, batch_size=BATCH_SIZE, allow_cpu=ALLOW_CPU_TRAINING,
    )
else:
    print("Training is disabled here. Completed trained models are indexed in models/retained_models.json.")


## Method summary

The model performs supervised binary road segmentation. MobileNetV2 supplies pretrained image features and the U-Net decoder learns a road probability per pixel. Skip connections restore spatial detail. The initial stage freezes the encoder, optimizes BCE plus soft Dice with Adam and selects checkpoints using validation road IoU. A later fixed-checkpoint evaluation reports 91.65% road IoU on all 689 real test images.

## Integration boundaries

TensorFlow/Keras supplies road perception. The current native traffic scenes use CARLA with rule-based vehicle controls and simulator states. Older MATLAB Hybrid A*/tracking examples are separate references. Camera perception remains observation-only; tracking, metric obstacle observations, selected destinations and perception-to-control integration require further implementation and evaluation.

A camera-view mask is in pixels. Calibrated geometry/range is required before it can support a map in metres. A visible-road label alone does not establish obstacle-free space or lane legality.

Original sequence IDs and the exact third-party binary-class conversion are unavailable. The 110 zero-mask exclusions are a recorded filtering decision, not proof that every excluded label was erroneous.

Read [model and training details](MODEL_AND_TRAINING.md), [results](docs/RESULTS.md) and [roadmap](docs/NEXT_STEPS.md).

References: [TensorFlow segmentation tutorial](https://www.tensorflow.org/tutorials/images/segmentation), [transfer learning and BatchNorm](https://www.tensorflow.org/guide/keras/transfer_learning), [GPU setup](https://www.tensorflow.org/install/pip).


## Historical first training run - 3 October 2026

Training ran separately through `train_idd.py` in WSL2 on the RTX 4060.
The source cells describe the original setup. Numerical evaluation records are documented separately; saved notebook execution output is omitted.

- Completed 18 epochs with early stopping in 88.1 minutes.
- Best checkpoint: `runs/frozen_01/best.keras`, from epoch 14.
- Validation global road IoU: **92.03%**; Dice: **95.85%**, threshold 0.5.
- Reloaded compiled evaluation and graph predictions reproduced these scores exactly.
- The encoder stayed frozen; the decoder learned.
- Split remains 5,507 training / 687 validation / 689 testing pairs.
- Images use the existing normalization once; masks stay 0/1. Test images were reserved during that run; the final locked evaluation is now complete.
- Later comparisons, final test evaluation and MATLAB traffic demonstrations are complete. Calibrated camera driving and RL remain unfinished.

[Full training report](runs/frozen_01/TRAINING_RESULT.md)

![Training curves](runs/frozen_01/training_curves.png)

[Validation prediction examples](runs/frozen_01/validation_review/validation_examples.png)

[Weakest validation cases](runs/frozen_01/validation_review/validation_lowest_iou.png)

Some weak cases have tiny or narrow provided road regions despite broader visible ground.
Confirm the original label definitions before changing the dataset or selecting further training.


## Completed locked-model test evaluation — 3 October 2026

All 689 test pairs were evaluated once with the fixed morning-best trial-13 epoch-1 model on the RTX 4060 in WSL2. Threshold 0.5, no training, augmentation or mask cleanup.

| Test metric | Result |
|---|---:|
| Road IoU | **91.65%** |
| Dice / road F1 | **95.65%** |
| Road precision | **95.19%** |
| Road recall | **96.11%** |
| Strict Boundary IoU, 5px | **55.23%** |
| Visible contour F1, 3px | **70.70%** |

Road IoU is 0.47 percentage points below validation. There are 14 images below 50% IoU and 42 below 75%; no test case was removed. Agreement with supplied masks is measured; sequence identities and exact class conversion remain unavailable.

Warm RGB-RAM-to-mask time: median 13.18 ms, p95 16.99 ms, excluding capture, file reading, tracking, planning and display. Saved binary predictions were independently recounted, and all model/data hashes remain unchanged. Original and boundary checkpoints were not tested.

[Full final report](runs/test_evaluation_20261003_morning_best/TEST_REPORT.md) · [All 689 test comparisons](runs/test_evaluation_20261003_morning_best/gallery/index.html) · [Per-image scores](runs/test_evaluation_20261003_morning_best/per_image_metrics.csv)

![Test evaluation summary](runs/test_evaluation_20261003_morning_best/plots/test_summary.png)

### Evaluation summary

We selected the checkpoint using validation data, then fixed its preprocessing and threshold before opening the test set. On all 689 test images it achieved 91.65% road IoU and 95.65% Dice. We checked boundaries, timing and weak cases, and verified saved predictions independently. This supports our segmentation prototype; driving safety still requires closed-loop simulation.

Reusable locked inference, separately evaluated road/detector models and native CARLA traffic scenes are implemented. MATLAB planning/tracking and replay examples remain separate references. Camera-driven control, selected destinations, tracking/range integration and a trained STOP / SLOW / PROCEED policy remain planned; see [the roadmap](docs/NEXT_STEPS.md) for current scope.
